# Synthetic Micromagnetic Landscape

This notebook uses lightweight synthetic LEM/NEB-like data to demonstrate the package data model and interactive PlotlyJS visualizations. The numbers are illustrative: they mimic minima, saddle energies, magnetic moments, and local magnetization textures without running a micromagnetic solver.

In [1]:
using Pkg
example_dir = basename(pwd()) == "examples" ? pwd() : joinpath(pwd(), "examples")
if !isfile(joinpath(example_dir, "Project.toml"))
    example_dir = pwd()
end
Pkg.activate(example_dir)
Pkg.develop(path=abspath(joinpath(example_dir, "..")))
Pkg.instantiate()

using DisconnectivityGraphs
using LinearAlgebra
using PlotlyJS
using Random
using Statistics

struct NotebookPlot{P}
    plot::P
end

function html_escape(value::AbstractString)
    return replace(value,
        "&" => "&amp;",
        "\"" => "&quot;",
        "<" => "&lt;",
        ">" => "&gt;")
end

Base.show(io::IO, ::MIME"text/plain", ::NotebookPlot) = print(io, "Plotly interactive figure")
Base.show(io::IO, mime::MIME"application/vnd.plotly.v1+json", value::NotebookPlot) = show(io, mime, value.plot)
function Base.show(io::IO, mime::MIME"text/html", value::NotebookPlot)
    html = sprint(show, mime, value.plot)
    print(io, """
    <iframe srcdoc="$(html_escape(html))"
            style="width:100%; height:680px; border:0;"
            sandbox="allow-scripts allow-same-origin">
    </iframe>
    """)
end

show_plotly(plot) = NotebookPlot(plot)

  Activating project at `~/Github/DisconnectivityGraphs.jl/examples`
   Resolving package versions...
    Updating `~/Github/DisconnectivityGraphs.jl/examples/Project.toml`
  [0defd51d] + DisconnectivityGraphs v0.1.0 `~/Github/DisconnectivityGraphs.jl`
  [f0f68f2c] + PlotlyJS v0.18.18
  [10745b16] + Statistics v1.11.1
  [37e2e46d] ~ LinearAlgebra ⇒ v1.11.0
  [9a3f8284] ~ Random ⇒ v1.11.0
    Updating `~/Github/DisconnectivityGraphs.jl/examples/Manifest.toml`
  [bf4720bc] + AssetRegistry v0.1.1
  [d1d4a3ce] + BitFlags v0.1.9
  [ad839575] + Blink v0.12.9
  [944b1d66] + CodecZlib v0.7.8
  [35d6a980] + ColorSchemes v3.31.0
  [3da002f7] + ColorTypes v0.12.1
  [c3611d14] + ColorVectorSpace v0.11.0
  [5ae59095] + Colors v0.13.1
  [f0e56b4a] + ConcurrentUtilities v2.5.1
  [9a962f9c] + DataAPI v1.16.0
  [e2d170a0] + DataValueInterfaces v1.0.0
  [8bb1440f] + DelimitedFiles v1.9.1
  [0defd51d] + DisconnectivityGraphs v0.1.0 `~/Github/DisconnectivityGraphs.jl`
  [ffbed154] + DocStringExtensions v0

show_plotly (generic function with 1 method)

## Synthetic Minima and Saddles

Each minimum stores an energy and a small amount of metadata. In a Merrill.jl adapter, this metadata would include grain id, mesh path, temperature, net magnetic moment, minimizer diagnostics, and a pointer to a saved node-magnetization state. Saddles represent NEB transition states connecting pairs of minima.

In [2]:
moments = Dict(
    :plus_x => [1.0, 0.05, 0.00],
    :tilted_plus => [0.75, 0.45, 0.10],
    :vortex_up => [0.05, 0.05, 0.55],
    :vortex_down => [-0.05, 0.02, -0.55],
    :tilted_minus => [-0.75, -0.35, -0.08],
    :minus_x => [-1.0, -0.03, 0.00],
)

minima = [
    Minimum(:plus_x, 0.00; metadata=Dict(:moment => moments[:plus_x], :texture => :uniform_plus)),
    Minimum(:tilted_plus, 0.18; metadata=Dict(:moment => moments[:tilted_plus], :texture => :flower_plus)),
    Minimum(:vortex_up, 0.42; metadata=Dict(:moment => moments[:vortex_up], :texture => :vortex_up)),
    Minimum(:vortex_down, 0.48; metadata=Dict(:moment => moments[:vortex_down], :texture => :vortex_down)),
    Minimum(:tilted_minus, 0.25; metadata=Dict(:moment => moments[:tilted_minus], :texture => :flower_minus)),
    Minimum(:minus_x, 0.05; metadata=Dict(:moment => moments[:minus_x], :texture => :uniform_minus)),
]

saddles = [
    Saddle(:plus_x, :tilted_plus, 1.35; metadata=Dict(:path => "NEB plus to tilted plus")),
    Saddle(:tilted_plus, :vortex_up, 1.85),
    Saddle(:vortex_up, :vortex_down, 2.70),
    Saddle(:vortex_down, :tilted_minus, 1.95),
    Saddle(:tilted_minus, :minus_x, 1.30),
    Saddle(:tilted_plus, :tilted_minus, 3.25),
]

landscape = LandscapeGraph(minima, saddles)
tree = disconnectivity_tree(landscape)
leaf_order(tree)

6-element Vector{Symbol}:
 :plus_x
 :tilted_plus
 :vortex_up
 :vortex_down
 :tilted_minus
 :minus_x

## Disconnectivity Tree

The branch height records the saddle energy where two previously disconnected basins first become connected. This view is useful for seeing whether the transition matrix is controlled by one dominant bottleneck or by many comparable pathways.

In [3]:
function plot_disconnectivity_tree(landscape; title="Synthetic disconnectivity tree")
    tree = disconnectivity_tree(landscape)
    layout = tree_layout(tree)
    xs = Float64[]
    ys = Float64[]
    for segment in layout.segments
        append!(xs, [segment.x0, segment.x1, NaN])
        append!(ys, [Float64(segment.y0), Float64(segment.y1), NaN])
    end

    minimum_by_id = Dict(m.id => m for m in landscape.minima)
    leaf_ids = leaf_order(tree)
    leaf_x = [layout.leaf_positions[id] for id in leaf_ids]
    leaf_y = [minimum_by_id[id].energy for id in leaf_ids]
    polarity = [dot(minimum_by_id[id].metadata[:moment], [1.0, 0.0, 0.0]) for id in leaf_ids]
    hover = ["$(id)<br>E=$(round(minimum_by_id[id].energy; digits=3))<br>m=$(round.(minimum_by_id[id].metadata[:moment]; digits=2))" for id in leaf_ids]

    traces = [
        scatter(x=xs, y=ys, mode="lines", line=attr(color="#334155", width=2.5), hoverinfo="skip", showlegend=false, name="branches"),
        scatter(x=leaf_x, y=leaf_y, mode="markers+text", text=String.(leaf_ids), textposition="bottom center",
            marker=attr(size=12, color=polarity, colorscale="RdBu", cmin=-1, cmax=1,
                colorbar=attr(title="m · x̂", x=1.03, xanchor="left", len=0.72), line=attr(color="white", width=1)),
            hovertext=hover, hoverinfo="text", showlegend=false, name="minima"),
    ]
    show_plotly(Plot(traces, Layout(title=title, xaxis=attr(visible=false), yaxis=attr(title="relative energy"),
        width=900, height=520, showlegend=false, margin=attr(l=60, r=130, t=70, b=55),
        plot_bgcolor="white", paper_bgcolor="white")))
end

plot_disconnectivity_tree(landscape)

Plotly interactive figure

## Synthetic 3D Magnetization States

This panel imitates the visual role of `plot_magnetization!` in Merrill.jl. The dropdown switches among synthetic LEM configurations while preserving the same grain geometry and camera.

In [4]:
grid = range(-1.0, 1.0; length=5)
points = [[x, y, z] for x in grid for y in grid for z in grid if maximum(abs.([x, y, z])) <= 1.0]

normalize3(v) = norm(v) == 0 ? [1.0, 0.0, 0.0] : v ./ norm(v)

function synthetic_vector(texture, p)
    x, y, z = p
    r2 = x^2 + y^2
    v = if texture == :uniform_plus
        [1.0, 0.04*y, 0.04*z]
    elseif texture == :uniform_minus
        [-1.0, 0.04*y, -0.04*z]
    elseif texture == :flower_plus
        [1.0, 0.22*y, 0.18*z]
    elseif texture == :flower_minus
        [-1.0, -0.18*y, -0.12*z]
    elseif texture == :vortex_up
        [-y, x, 0.55 * exp(-3r2)]
    elseif texture == :vortex_down
        [y, -x, -0.55 * exp(-3r2)]
    else
        [1.0, 0.0, 0.0]
    end
    normalize3(v)
end

synthetic_vectors(texture, points) = [synthetic_vector(texture, p) for p in points]

function synthetic_helicity(texture, p; h=0.08)
    basis = ([h, 0.0, 0.0], [0.0, h, 0.0], [0.0, 0.0, h])
    dm_dx = (synthetic_vector(texture, p .+ basis[1]) .- synthetic_vector(texture, p .- basis[1])) ./ (2h)
    dm_dy = (synthetic_vector(texture, p .+ basis[2]) .- synthetic_vector(texture, p .- basis[2])) ./ (2h)
    dm_dz = (synthetic_vector(texture, p .+ basis[3]) .- synthetic_vector(texture, p .- basis[3])) ./ (2h)
    curl_m = [dm_dy[3] - dm_dz[2], dm_dz[1] - dm_dx[3], dm_dx[2] - dm_dy[1]]
    return dot(synthetic_vector(texture, p), curl_m)
end

function helicity_color_values(raw_values; scale_quantile=0.9, display_quantiles=(0.02, 0.98))
    values = Float64.(raw_values)
    isempty(values) && return Float64[]
    scale = max(quantile(abs.(values), scale_quantile), eps(Float64))
    compressed = asinh.(values ./ scale)
    lo = quantile(compressed, display_quantiles[1])
    hi = quantile(compressed, display_quantiles[2])
    hi > lo || return zeros(Float64, length(values))
    normalized = (clamp.(compressed, lo, hi) .- lo) ./ (hi - lo)
    return 2.0 .* normalized .- 1.0
end

function helicity_scaled_cone_components(vecs, helicity_values; arrow_length=0.23, min_arrow_fraction=0.8)
    colors = Float64.(helicity_values)
    if isempty(colors)
        color01 = Float64[]
    elseif minimum(colors) < 0.0 < maximum(colors)
        max_abs = max(maximum(abs.(colors)), eps(Float64))
        color01 = 0.5 .+ 0.5 .* colors ./ max_abs
    else
        span = max(maximum(colors) - minimum(colors), eps(Float64))
        color01 = (colors .- minimum(colors)) ./ span
    end
    magnitudes = arrow_length .* (min_arrow_fraction .+ (1.0 - min_arrow_fraction) .* color01)
    scaled = [normalize3(v) .* mag for (v, mag) in zip(vecs, magnitudes)]
    cmin = arrow_length * min_arrow_fraction
    cmax = arrow_length
    tickvals = cmin .+ (cmax - cmin) .* [0.0, 0.5, 1.0]
    ticktext = ["-", "0", "+"]
    return scaled, cmin, cmax, tickvals, ticktext
end

function cone_trace_for_minimum(minimum; visible=false)
    texture = minimum.metadata[:texture]
    vecs = synthetic_vectors(texture, points)
    raw_helicity = [synthetic_helicity(texture, p) for p in points]
    helicity_values = helicity_color_values(raw_helicity)
    scaled_vecs, cmin, cmax, tickvals, ticktext = helicity_scaled_cone_components(vecs, helicity_values)
    cone(x=[p[1] for p in points], y=[p[2] for p in points], z=[p[3] for p in points],
         u=[v[1] for v in scaled_vecs], v=[v[2] for v in scaled_vecs], w=[v[3] for v in scaled_vecs],
         sizemode="raw", sizeref=1.0, anchor="tail", colorscale="RdBu", reversescale=true,
         cmin=cmin, cmax=cmax,
         colorbar=attr(title=attr(text="helicity", side="bottom"), tickvals=tickvals, ticktext=ticktext,
             thickness=15, len=0.72, x=0.92, xanchor="left", y=0.5, yanchor="middle"),
         showscale=true, visible=visible, name=String(minimum.id))
end

function plot_synthetic_state_dropdown(minima)
    traces = [cone_trace_for_minimum(m; visible=i == 1) for (i, m) in pairs(minima)]
    buttons = [attr(label=String(m.id), method="update",
        args=[attr(visible=[j == i for j in eachindex(minima)]),
              attr(title="Synthetic magnetization: $(m.id)")]) for (i, m) in pairs(minima)]
    layout = Layout(title="Synthetic magnetization: $(minima[1].id)", width=820, height=650,
        scene=attr(aspectmode="data", xaxis=attr(visible=false), yaxis=attr(visible=false), zaxis=attr(visible=false)),
        margin=attr(l=10, r=110, t=70, b=10), showlegend=false,
        updatemenus=[attr(type="dropdown", x=0.02, y=1.08, buttons=buttons)])
    show_plotly(Plot(traces, layout))
end

plot_synthetic_state_dropdown(minima)

Plotly interactive figure